In [1]:
#Import all necessary datasets, libraries, and packages
!pip install nba_api
import pandas as pd
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.endpoints import boxscoreadvancedv3
from nba_api.stats.static import players
from nba_api.stats.static import teams
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.3/285.3 kB 3.9 MB/s eta 0:00:00


In [2]:
#Finds a player's game by game performance data from nba_api
def get_player_data(player_name, season_type, season):
    player_list = players.get_players()
    player = next((p for p in player_list if player_name.lower() in p['full_name'].lower()), None)
    if not player:
        raise ValueError(f"Player '{player_name}' not found.")
    log = playergamelog.PlayerGameLog(player_id=player['id'], season=season, season_type_all_star=season_type)
    df = log.get_data_frames()[0]

    #Add player to dataframe and return
    df['PLAYER_NAME'] = player_name
    return df

Here, the opponent’s defensive rating is pulled from each game and attached to the dataset.


In [3]:
#Adds defensive rating to the dataframe
def get_player_logs_with_opp_def(player_id, season_type, season):
    log_df = playergamelog.PlayerGameLog(
        player_id=player_id,
        season=season,
        season_type_all_star=season_type
    ).get_data_frames()[0]

    #For each game, extract the opponent abbreviation from MATCHUP and lookup the associated drtg
    opp_drtg = []
    for _, row in log_df.iterrows():
        matchup = row['MATCHUP']
        parts = matchup.split()
        if len(parts) < 3:
            opp_drtg.append(None)
            continue
        opponent_abbr = parts[2]
        if season_type.lower() == 'playoffs':
            rating = playoffs_drtg.get(opponent_abbr)
        else:
            rating = regular_season_drtg.get(opponent_abbr)
        opp_drtg.append(rating)
    log_df['OPP_DEF_RATING'] = opp_drtg
    return log_df.dropna(subset=['OPP_DEF_RATING'])

This section computes rolling averages for the target stat (PTS, REB, or AST) and minutes played, adds rest days, flags home games, and calculates how aggressive the line is compared to recent form LINE_DIFF. It also creates the target variable OVER, which is used to train the model.

In [4]:
#Preparing features
def prepare_features(df, stat_col='PTS', line=None): #defaults
    df = df.sort_values(by='GAME_DATE')
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df[f'{stat_col}_ROLL5'] = df[stat_col].rolling(window=5).mean()
    df['MIN_ROLL5'] = df['MIN'].rolling(window=5).mean()
    df['REST_DAYS'] = df['GAME_DATE'].diff().dt.days.fillna(0)
    df['IS_HOME'] = df['MATCHUP'].apply(lambda x: 1 if 'vs.' in x else 0)
    df['LINE'] = line #line is always 0 due to dependence on line_diff, but the model needs it to avoid error when generating the over label.
    df['LINE_DIFF'] = df['LINE'] - df[f'{stat_col}_ROLL5']
    df['OVER'] = (df[stat_col] > df['LINE']).astype(int)
    return df.dropna()

Here the model is trained evluation metrics are calculated. Probability of the predicted outcome is also calculated here.


In [5]:
def train_and_predict(df, features, stat_col='PTS'): #defaults again

    #Standard logreg training methods here
    X = df[features].copy()
    y = df['OVER'].copy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = LogisticRegression(random_state=42, max_iter=1000) #ensure convergence
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    #calculate evaluation metrics and coefficients of logreg equation
    acc = accuracy_score(y_test, y_pred)
    Cmatrix = confusion_matrix(y_test, y_pred)
    coeffs = model.coef_[0]  # shape (1, n_features)
    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': coeffs
    }).sort_values(by='Coefficient', key=abs, ascending=False)
    coeffs=coef_df

    #Construct one row df for next game based on most recent vals
    latest = df.iloc[-1]
    next_input = pd.DataFrame([{
    'LINE': latest['LINE'],
    f'{stat_col}_ROLL5': latest[f'{stat_col}_ROLL5'],
    'MIN_ROLL5': latest['MIN_ROLL5'],
    'REST_DAYS': rest_days,
    'IS_HOME': home,
    'LINE_DIFF': latest['LINE_DIFF'],
    'OPP_DEF_RATING': opp_drtg
    }])

    #scale input features and predict prob of going over
    next_input_scaled = scaler.transform(next_input)
    prob = model.predict_proba(next_input_scaled)[0][1]
    return acc, Cmatrix, coeffs, prob

In [6]:
#tried to pull drtg directly from api but it kept timing out so I just made the libraries
regular_season_drtg = {
    'OKC': 106.6,
    'ORL': 109.1,
    'LAC': 109.4,
    'BOS': 110.1,
    'HOU': 110.3,
    'MIN': 110.8,
    'GSW': 111.0,
    'CLE': 111.1,
    'MIA': 112.0,
    'DET': 112.5,
    'MEM': 112.6,
    'MIL': 112.7,
    'NYK': 112.9,
    'IND': 113.2,
    'TOR': 113.4,
    'POR': 113.7,
    'LAL': 113.8,
    'ATL': 114.8,
    'CHI': 114.8,
    'DAL': 115.0,
    'DEN': 115.1,
    'SAC': 115.3,
    'BKN': 115.4,
    'CHA': 115.7,
    'SAS': 116.3,
    'PHI': 117.3,
    'PHX': 117.4,
    'WAS': 118.0,
    'NOP': 119.1,
    'UTA': 119.4
}
playoffs_drtg = {
    'OKC': 105.7,
    'BOS': 108.1,
    'DET': 109.8,
    'MIN': 109.9,
    'CLE': 110.7,
    'HOU': 111.7,
    'GSW': 112.0,
    'NYK': 113.2,
    'IND': 113.5,
    'DEN': 114.8,
    'LAC': 115.3,
    'LAL': 115.6,
    'MEM': 117.4,
    'ORL': 118.0,
    'MIL': 119.4,
    'MIA': 136.2
}

In [7]:
season='2024-25'

In [8]:
season_type = input("Enter Regular Season or Playoffs: ")

Enter Regular Season or Playoffs: Playoffs


In [9]:
opp = input("Enter opponent's 3-letter abbreviation: ").upper()
if season_type.lower() == 'playoffs':
    opp_drtg = playoffs_drtg.get(opp)
else:
    opp_drtg = regular_season_drtg.get(opp)

Enter opponent's 3-letter abbreviation: IND


In [10]:
rest_days = int(input("Enter the number of rest days before the next game: "))

Enter the number of rest days before the next game: 2


In [11]:
#nba_api uses 1 for home game and 0 for away
home = int(input("Is the next game a home game? 1 for yes, 0 for no: "))

Is the next game a home game? 1 for yes, 0 for no: 0


In [12]:
player_name = input("Enter NBA player name: ")

Enter NBA player name: Shai Gilgeous-Alexander


In [13]:
stat = input("Enter stat to predict over for (PTS/REB/AST): ").upper()

Enter stat to predict over for (PTS/REB/AST): PTS


In [14]:
line = float(input(f"Enter the betting line for {stat}: "))

Enter the betting line for PTS: 32.5


Here everything is run and outputted to the terminal.

In [15]:
#finds matching player from nba_api and the respective drtgs for games played
player_list = players.get_players()
player = next((p for p in player_list if player_name.lower() in p['full_name'].lower()), None)
df = get_player_logs_with_opp_def(player['id'], season=season, season_type=season_type)
df = prepare_features(df, stat_col=stat, line=line)

#Define features return metrics from train_and_predict
features = ['LINE', f'{stat}_ROLL5', 'MIN_ROLL5', 'REST_DAYS', 'IS_HOME', 'LINE_DIFF', 'OPP_DEF_RATING']
acc, Cmatrix, coeffs, prob = train_and_predict(df, features, stat_col=stat)

print(f"Accuracy Score: {acc:.2f}")
print(Cmatrix)
print(coeffs)
print(f"Predicted probability of going OVER {line} {stat}: {prob:.2%}")

Accuracy Score: 0.75
[[1 1]
 [0 2]]
          Feature  Coefficient
4         IS_HOME     0.740195
5       LINE_DIFF    -0.438588
1       PTS_ROLL5     0.438588
2       MIN_ROLL5    -0.423787
6  OPP_DEF_RATING     0.200038
3       REST_DAYS     0.004498
0            LINE     0.000000
Predicted probability of going OVER 32.5 PTS: 62.32%
